Студент: Гапуніч Анатолій

Група: AI&ML 4

# Домашня робота 2 - Завантаження даних та основи SQL. DQL-команди (Тема 3)

**Обраний датасет:** UCI Adult / Census Income, офіційний файл `adult.data` (тренувальна частина, 32 561 рядок, 15 колонок).

**Мета:** відтворювано отримати CSV з офіційного джерела, завантажити його через PostgreSQL 16 `COPY FROM STDIN` через Psycopg 3 у staging, очистити й типізувати дані, виконати data-quality audit та 10 DQL/EDA-запитів.

Датасет описує записи перепису населення США 1994 року; цільова задача оригінального набору — класифікація річного доходу (`>50K` проти `<=50K`). Офіційна сторінка: <https://archive.ics.uci.edu/dataset/2/adult>. Посилання на вихідний файл: <https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data>. Ліцензія набору — CC BY 4.0; бібліографічне посилання: Becker, B. & Kohavi, R. (1996), DOI: <https://doi.org/10.24432/C5XW20>.

**Важливе рішення щодо scope:** використовуємо лише `adult.data`, не об'єднуємо його з `adult.test`. Так ми зберігаємо одну однорідну частину набору, у якій немає окремого формату target з крапкою наприкінці, і маємо понад мінімальні 30 000 рядків.


## 1. Setup і перевірка PostgreSQL 16

Встановлюємо PostgreSQL 16 із офіційного PGDG APT-репозиторію, запускаємо локальний кластер і підключаємося через Unix socket за допомогою Psycopg 3. SQL виконується двома допоміжними функціями: `execute_script()` — для DDL та багатокомандних скриптів; `query_df()` — для `SELECT` із поверненням результату в pandas DataFrame.


In [ ]:
%%bash
set -euo pipefail

# Install prerequisites and add the official PostgreSQL APT repository.
apt-get update -qq
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq ca-certificates curl
install -d /usr/share/postgresql-common/pgdg
curl -fsSL https://www.postgresql.org/media/keys/ACCC4CF8.asc \
  -o /usr/share/postgresql-common/pgdg/apt.postgresql.org.asc

CODENAME="$(. /etc/os-release && printf '%s' "${VERSION_CODENAME:-}")"
ARCH="$(dpkg --print-architecture)"
if [[ -z "${CODENAME}" ]]; then
  echo "Не вдалося визначити VERSION_CODENAME із /etc/os-release." >&2
  exit 1
fi

cat > /etc/apt/sources.list.d/pgdg.sources <<EOF
Types: deb
URIs: https://apt.postgresql.org/pub/repos/apt
Suites: ${CODENAME}-pgdg
Architectures: ${ARCH}
Components: main
Signed-By: /usr/share/postgresql-common/pgdg/apt.postgresql.org.asc
EOF

apt-get update -qq
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq postgresql-16 postgresql-client-16

# Start PostgreSQL 16. If it is already running, verify its status instead.
pg_ctlcluster 16 main start || pg_ctlcluster 16 main status

# Match the Colab root OS user with a PostgreSQL role for local peer auth.
if ! runuser -u postgres -- psql -tAc "SELECT 1 FROM pg_roles WHERE rolname='root'" | grep -q '^1$'; then
  runuser -u postgres -- createuser --superuser root
fi

python -m pip install -q "psycopg[binary]" pandas

# Print the installed PostgreSQL server version.
runuser -u postgres -- psql -tAc "SHOW server_version;"


Preconfiguring packages ...
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../ca-certificates_20260816ubuntu0.24.04.2_all.deb ...
Unpacking ca-certificates (20260816ubuntu0.24.04.2) over (20260816ubuntu0.24.04.1) ...
Setting up ca-certificates (20260816ubuntu0.24.04.2) ...
Updating certificates in /etc/ssl/certs...
0 added, 0 removed; done.
Running hooks in /etc/ca-certificates/update.d...
done.
Processing triggers for man-db (2.12.0-4build2) ...
Processing triggers for ca-certificates-java (20240118) ...
[0.013s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.013s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
done.
Preconfiguring packages ...
Selecting previously unselected package libjson-perl.
(Reading database ... 122

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
import hashlib
import time
from pathlib import Path

import pandas as pd
import psycopg
from IPython.display import display, Markdown

# Підключаємося до локального PostgreSQL 16 через Unix socket.
conn = psycopg.connect(
    dbname="postgres",
    user="root",
    host="/var/run/postgresql",
    connect_timeout=10,
)
conn.autocommit = False


def timed_step(name: str, operation):
    """Виконати операцію з видимими часовими мітками та вимірюванням часу."""
    started_at = time.perf_counter()
    print(f"[START] {name}", flush=True)
    try:
        result = operation()
    except Exception as exc:
        elapsed = time.perf_counter() - started_at
        print(
            f"[FAILED] {name} після {elapsed:.2f} сек. "
            f"Тип помилки: {type(exc).__name__}: {exc}",
            flush=True,
        )
        raise
    elapsed = time.perf_counter() - started_at
    print(f"[DONE] {name}: {elapsed:.2f} сек.", flush=True)
    return result


def execute_script(sql_script: str) -> None:
    """Виконати DDL або багатокомандний SQL-скрипт однією транзакцією."""
    try:
        with conn.cursor() as cur:
            cur.execute(sql_script, prepare=False)
        conn.commit()
    except Exception:
        conn.rollback()
        raise


def query_df(sql_query: str) -> pd.DataFrame:
    """Виконати SELECT і повернути результат у pandas DataFrame.

    Після читання явно завершуємо read-транзакцію, щоб не залишати
    з'єднання в стані idle in transaction між комірками notebook.
    """
    try:
        with conn.cursor() as cur:
            cur.execute(sql_query, prepare=False)
            if cur.description is None:
                raise ValueError(
                    "query_df() очікує запит, що повертає рядки (наприклад, SELECT)."
                )
            rows = cur.fetchall()
            columns = [column.name for column in cur.description]
        conn.commit()
        return pd.DataFrame(rows, columns=columns)
    except Exception:
        conn.rollback()
        raise


# Перевіряємо серверну версію, номер major-версії та timezone сесії.
version = query_df("""
    SELECT current_setting('server_version') AS postgresql_version,
           current_setting('server_version_num') AS version_num,
           current_setting('TimeZone') AS session_time_zone
""")
display(version)
major_version = int(version.loc[0, "version_num"]) // 10000
assert major_version == 16, (
    f"За вимогами потрібен PostgreSQL 16, отримано {version.loc[0, 'postgresql_version']}"
)
print("Підключення до PostgreSQL 16 через Psycopg 3 встановлено.")

,postgresql_version,version_num,session_time_zone
0,16.15 (Ubuntu 16.15-1.pgdg24.04+2),160015,Etc/UTC


Підключення до PostgreSQL 16 через Psycopg 3 встановлено.


## 2. Отримання офіційного датасету та контрольований CSV

`adult.data` не має заголовка, а значення розділені комами з пробілами після роздільника. Під час читання явно задаємо назви 15 колонок, зберігаємо всі значення як текст і вимикаємо автоматичне перетворення `?` на пропуски. Після цього обрізаємо зайві пробіли та експортуємо CSV із заголовком для PostgreSQL `COPY`.

In [ ]:
DATA_URL = (
    'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
)
CSV_PATH = Path('/content/hw3_adult.csv')

COLUMNS = [
    'age', 'workclass', 'fnlwgt', 'education', 'education_num',
    'marital_status', 'occupation', 'relationship', 'race', 'sex',
    'capital_gain', 'capital_loss', 'hours_per_week', 'native_country',
    'income'
]

# keep_default_na=False важливий: маркер '?' має залишитися сирим текстом,
# щоб його нормалізацію можна було виконати прозоро в SQL.
df = pd.read_csv(
    DATA_URL,
    header=None,
    names=COLUMNS,
    skipinitialspace=True,
    dtype='string',
    keep_default_na=False,
)

# Нормалізуємо лише пробіли, не змінюючи спеціальний маркер '?'.
for column in COLUMNS:
    df[column] = df[column].str.strip()

df = df.loc[~df.eq('').all(axis=1)].reset_index(drop=True)

assert len(df) >= 30_000, f'Очікувалося щонайменше 30 000 рядків, отримано {len(df)}'
assert list(df.columns) == COLUMNS

CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(CSV_PATH, index=False, encoding='utf-8', na_rep='')

csv_sha256 = hashlib.sha256(CSV_PATH.read_bytes()).hexdigest()
print(f'Розмір DataFrame: {df.shape[0]:,} рядків × {df.shape[1]} колонок')
print(f'CSV: {CSV_PATH}')
print(f'SHA-256 сформованого CSV: {csv_sha256}')
display(df.head())

Розмір DataFrame: 32,561 рядків × 15 колонок
CSV: /content/hw3_adult.csv
SHA-256 сформованого CSV: 3b8a6abd697a6623ef2ccbffc3e2802e167e7fdaa853003d3bd557b0ce7f5d2a


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


## 3. Staging-таблиця та `COPY FROM STDIN`

Усі 15 полів staging мають тип `TEXT`. На цьому рівні не виконуємо числове приведення й не втрачаємо оригінальний маркер `?`.

Імпорт розбитий на вимірювані етапи: налаштування таймаутів PostgreSQL, створення staging, передавання CSV через Psycopg 3 `COPY` API та контроль `COUNT(*)`. Прогрес і тривалість кожного етапу друкуються одразу. Очікування блокування обмежено 15 секундами, а максимальний час однієї SQL-команди — 180 секундами.

### Опціональне очищення таблиць

Ця комірка видаляє лише `hw3_adult` і `hw3_adult_staging` (`CASCADE`) — на випадок, якщо потрібно перезавантажити дані з нуля.


In [ ]:
# Видаляє таблиці цього завдання; перед DROP завершуємо можливу незавершену транзакцію.

def drop_homework_tables():
    if conn.closed:
        raise RuntimeError("З'єднання PostgreSQL закрите. Повторно виконайте комірку підключення.")

    conn.rollback()
    execute_script("""
        DROP TABLE IF EXISTS hw3_adult CASCADE;
        DROP TABLE IF EXISTS hw3_adult_staging CASCADE;
    """)


timed_step("Повне очищення таблиць hw3_adult та hw3_adult_staging", drop_homework_tables)
print("Готово: обидві таблиці цього домашнього завдання видалено.")

[START] Повне очищення таблиць hw3_adult та hw3_adult_staging
[DONE] Повне очищення таблиць hw3_adult та hw3_adult_staging: 0.01 сек.
Готово: обидві таблиці цього домашнього завдання видалено.


In [ ]:
# Перед DDL задаємо обмеження очікування блокувань і тривалості SQL-команд.
# Це запобігає багатохвилинному/багатогодинному очікуванню при завислому lock.
def configure_sql_timeouts():
    conn.rollback()
    with conn.cursor() as cur:
        cur.execute("SET lock_timeout = '15s'")
        cur.execute("SET statement_timeout = '180s'")
    conn.commit()


timed_step("Налаштування SQL timeouts (lock=15s, statement=180s)", configure_sql_timeouts)

# Створюємо staging заново. Фінальну таблицю hw3_adult ця комірка не видаляє.
def create_staging_table():
    execute_script("""
        DROP TABLE IF EXISTS hw3_adult_staging;

        CREATE TABLE hw3_adult_staging (
            age_raw            TEXT,
            workclass_raw      TEXT,
            fnlwgt_raw         TEXT,
            education_raw      TEXT,
            education_num_raw  TEXT,
            marital_status_raw TEXT,
            occupation_raw     TEXT,
            relationship_raw   TEXT,
            race_raw           TEXT,
            sex_raw            TEXT,
            capital_gain_raw   TEXT,
            capital_loss_raw   TEXT,
            hours_per_week_raw TEXT,
            native_country_raw TEXT,
            income_raw         TEXT
        );
    """)


timed_step("DROP/CREATE staging-таблиці", create_staging_table)

copy_sql = """
COPY hw3_adult_staging (
    age_raw, workclass_raw, fnlwgt_raw, education_raw,
    education_num_raw, marital_status_raw, occupation_raw,
    relationship_raw, race_raw, sex_raw, capital_gain_raw,
    capital_loss_raw, hours_per_week_raw, native_country_raw, income_raw
)
FROM STDIN
WITH (FORMAT CSV, HEADER TRUE, DELIMITER ',', NULL '')
"""


def copy_csv_to_staging():
    if not Path(CSV_PATH).is_file():
        raise FileNotFoundError(
            f"CSV не знайдено: {CSV_PATH}. Спочатку виконайте комірку отримання датасету."
        )

    total_bytes = Path(CSV_PATH).stat().st_size
    bytes_sent = 0
    chunk_size = 1024 * 1024  # 1 MiB

    try:
        # Для cursor.copy() передаємо байтові блоки; CSV уже закодований у UTF-8.
        with open(CSV_PATH, 'rb') as csv_file:
            with conn.cursor() as cur:
                with cur.copy(copy_sql) as copy:
                    while True:
                        chunk = csv_file.read(chunk_size)
                        if not chunk:
                            break
                        copy.write(chunk)
                        bytes_sent += len(chunk)
                        percent = 100 * bytes_sent / total_bytes if total_bytes else 100
                        print(
                            f"  COPY progress: {bytes_sent:,}/{total_bytes:,} bytes "
                            f"({percent:.1f}%)",
                            flush=True,
                        )

        # Фіксуємо імпорт тільки після успішного завершення COPY.
        conn.commit()
        return bytes_sent
    except Exception:
        conn.rollback()
        raise


bytes_copied = timed_step("COPY FROM STDIN (передавання CSV у PostgreSQL)", copy_csv_to_staging)
print(f"Передано до PostgreSQL: {bytes_copied:,} bytes")

staging_count = timed_step(
    "Перевірка COUNT(*) у staging",
    lambda: query_df("SELECT COUNT(*) AS loaded_rows FROM hw3_adult_staging"),
)
display(staging_count)

loaded_rows = int(staging_count.loc[0, "loaded_rows"])
expected_rows = len(df)
assert loaded_rows == expected_rows, (
    f"Кількість рядків не збігається: у CSV {expected_rows:,}, "
    f"у staging {loaded_rows:,}. Перевірте імпорт."
)
print(f"Успішно: у staging завантажено всі {loaded_rows:,} рядків.")

[START] Налаштування SQL timeouts (lock=15s, statement=180s)
[DONE] Налаштування SQL timeouts (lock=15s, statement=180s): 0.00 сек.
[START] DROP/CREATE staging-таблиці
[DONE] DROP/CREATE staging-таблиці: 0.03 сек.
[START] COPY FROM STDIN (передавання CSV у PostgreSQL)
  COPY progress: 1,048,576/3,518,606 bytes (29.8%)
  COPY progress: 2,097,152/3,518,606 bytes (59.6%)
  COPY progress: 3,145,728/3,518,606 bytes (89.4%)
  COPY progress: 3,518,606/3,518,606 bytes (100.0%)
[DONE] COPY FROM STDIN (передавання CSV у PostgreSQL): 0.30 сек.
Передано до PostgreSQL: 3,518,606 bytes
[START] Перевірка COUNT(*) у staging
[DONE] Перевірка COUNT(*) у staging: 0.02 сек.


,loaded_rows
0,32561


Успішно: у staging завантажено всі 32,561 рядків.


## 4. Типізована таблиця та правила очищення

### Обґрунтування типів

| Колонка | Семантика та джерельний формат | Тип у clean | Очищення |
|---|---|---|---|
| `age` | Вік у повних роках, ціле число у CSV | `INTEGER NOT NULL` | Перевірка, що рядок містить цифри; предметний діапазон 17–100 |
| `workclass` | Категорія зайнятості | `TEXT NULL` | `TRIM`; `?` і порожнє значення → `NULL` |
| `fnlwgt` | Census final weight, ціле число | `BIGINT NOT NULL` | Числовий парсинг; значення має бути більше нуля |
| `education` | Назва рівня освіти | `TEXT NOT NULL` | `TRIM`; порожнє значення не допускається |
| `education_num` | Числове кодування освіти | `SMALLINT NOT NULL` | Парсинг цілого; діапазон 1–16 |
| `marital_status` | Сімейний стан, категорія | `TEXT NOT NULL` | `TRIM`; порожнє значення не допускається |
| `occupation` | Професія / тип роботи | `TEXT NULL` | `TRIM`; `?` і порожнє значення → `NULL` |
| `relationship` | Категорія сімейного зв'язку | `TEXT NOT NULL` | `TRIM`; порожнє значення не допускається |
| `race` | Категорія race з джерела | `TEXT NOT NULL` | `TRIM`; зберігаємо без перекодування |
| `sex` | Двійкова категорія джерела | `TEXT NOT NULL` | Дозволені значення `Female` / `Male`; не перетворюємо на boolean, бо це категорія, а не універсальна логічна ознака |
| `capital_gain` | Значення приросту капіталу, цілі числа | `INTEGER NOT NULL` | Парсинг цілого; значення не від'ємне |
| `capital_loss` | Значення втрат капіталу, цілі числа | `INTEGER NOT NULL` | Парсинг цілого; значення не від'ємне |
| `hours_per_week` | Кількість робочих годин на тиждень | `SMALLINT NOT NULL` | Парсинг цілого; фізично можливий діапазон 1–168 |
| `native_country` | Країна народження, категорія | `TEXT NULL` | `TRIM`; `?` і порожнє значення → `NULL` |
| `income` | Target: `>50K` або `<=50K` | `BOOLEAN NOT NULL` (`income_over_50k`) | `>50K` → `TRUE`; `<=50K` → `FALSE`; інші значення не завантажуються |

Технічний `id` — identity primary key, він не є business key. `TEXT` використано для категорій без заданого бізнес-ліміту довжини. `NUMERIC(p,s)` не потрібен, оскільки в цьому датасеті немає грошових колонок із дробовими значеннями. Так само немає подій із часом, тому `DATE` та `TIMESTAMPTZ` тут семантично недоречні.

У таблиці використано більше двох `CHECK`: вони обмежують віковий діапазон, вагу, шкалу освіти, допустиму категорію `sex`, невід'ємні показники капіталу та діапазон годин.

In [ ]:
execute_script("""
DROP TABLE IF EXISTS hw3_adult CASCADE;

CREATE TABLE hw3_adult (
    id                BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    age               INTEGER NOT NULL CHECK (age BETWEEN 17 AND 100),
    workclass         TEXT,
    fnlwgt            BIGINT NOT NULL CHECK (fnlwgt > 0),
    education         TEXT NOT NULL CHECK (BTRIM(education) <> ''),
    education_num     SMALLINT NOT NULL CHECK (education_num BETWEEN 1 AND 16),
    marital_status    TEXT NOT NULL CHECK (BTRIM(marital_status) <> ''),
    occupation        TEXT,
    relationship      TEXT NOT NULL CHECK (BTRIM(relationship) <> ''),
    race              TEXT NOT NULL CHECK (BTRIM(race) <> ''),
    sex               TEXT NOT NULL CHECK (sex IN ('Female', 'Male')),
    capital_gain      INTEGER NOT NULL CHECK (capital_gain >= 0),
    capital_loss      INTEGER NOT NULL CHECK (capital_loss >= 0),
    hours_per_week    SMALLINT NOT NULL CHECK (hours_per_week BETWEEN 1 AND 168),
    native_country    TEXT,
    income_over_50k   BOOLEAN NOT NULL
);

WITH parsed AS (
    SELECT
        CASE WHEN BTRIM(age_raw) ~ '^[0-9]+$'
             THEN BTRIM(age_raw)::INTEGER END AS age,
        NULLIF(NULLIF(BTRIM(workclass_raw), ''), '?') AS workclass,
        CASE WHEN BTRIM(fnlwgt_raw) ~ '^[0-9]+$'
             THEN BTRIM(fnlwgt_raw)::BIGINT END AS fnlwgt,
        NULLIF(BTRIM(education_raw), '') AS education,
        CASE WHEN BTRIM(education_num_raw) ~ '^[0-9]+$'
             THEN BTRIM(education_num_raw)::SMALLINT END AS education_num,
        NULLIF(BTRIM(marital_status_raw), '') AS marital_status,
        NULLIF(NULLIF(BTRIM(occupation_raw), ''), '?') AS occupation,
        NULLIF(BTRIM(relationship_raw), '') AS relationship,
        NULLIF(BTRIM(race_raw), '') AS race,
        NULLIF(BTRIM(sex_raw), '') AS sex,
        CASE WHEN BTRIM(capital_gain_raw) ~ '^[0-9]+$'
             THEN BTRIM(capital_gain_raw)::INTEGER END AS capital_gain,
        CASE WHEN BTRIM(capital_loss_raw) ~ '^[0-9]+$'
             THEN BTRIM(capital_loss_raw)::INTEGER END AS capital_loss,
        CASE WHEN BTRIM(hours_per_week_raw) ~ '^[0-9]+$'
             THEN BTRIM(hours_per_week_raw)::SMALLINT END AS hours_per_week,
        NULLIF(NULLIF(BTRIM(native_country_raw), ''), '?') AS native_country,
        CASE BTRIM(income_raw)
            WHEN '>50K' THEN TRUE
            WHEN '<=50K' THEN FALSE
            ELSE NULL
        END AS income_over_50k
    FROM hw3_adult_staging
)
INSERT INTO hw3_adult (
    age, workclass, fnlwgt, education, education_num,
    marital_status, occupation, relationship, race, sex,
    capital_gain, capital_loss, hours_per_week, native_country,
    income_over_50k
)
SELECT
    age, workclass, fnlwgt, education, education_num,
    marital_status, occupation, relationship, race, sex,
    capital_gain, capital_loss, hours_per_week, native_country,
    income_over_50k
FROM parsed
WHERE age BETWEEN 17 AND 100
  AND fnlwgt > 0
  AND education IS NOT NULL
  AND education_num BETWEEN 1 AND 16
  AND marital_status IS NOT NULL
  AND relationship IS NOT NULL
  AND race IS NOT NULL
  AND sex IN ('Female', 'Male')
  AND capital_gain >= 0
  AND capital_loss >= 0
  AND hours_per_week BETWEEN 1 AND 168
  AND income_over_50k IS NOT NULL
""")

clean_count = query_df("SELECT COUNT(*) AS clean_rows FROM hw3_adult")
display(clean_count)
print('Типізовану таблицю hw3_adult створено та заповнено.')


,clean_rows
0,32561


Типізовану таблицю hw3_adult створено та заповнено.


## 5. Завдання 3 — Data-quality audit

### 3.1. Кількість рядків у кожному шарі

In [ ]:
row_audit = query_df('''
    SELECT 'staging' AS layer, COUNT(*) AS n_rows FROM hw3_adult_staging
    UNION ALL
    SELECT 'clean' AS layer, COUNT(*) AS n_rows FROM hw3_adult
    UNION ALL
    SELECT 'rejected' AS layer,
           (SELECT COUNT(*) FROM hw3_adult_staging)
           - (SELECT COUNT(*) FROM hw3_adult) AS n_rows
    ORDER BY layer
''')
display(row_audit)

counts = dict(zip(row_audit['layer'], row_audit['n_rows']))
print(
    f"У staging завантажено {counts['staging']:,} рядків; "
    f"у clean залишено {counts['clean']:,}; "
    f"відхилено {counts['rejected']:,}. "
    "Відхиляються записи з некоректними/відсутніми обов'язковими полями "
    "або порушенням заданих доменних діапазонів. '?' у трьох nullable-полях "
    "не є причиною відхилення — він нормалізується до SQL NULL."
)

,layer,n_rows
0,clean,32561
1,rejected,0
2,staging,32561


У staging завантажено 32,561 рядків; у clean залишено 32,561; відхилено 0. Відхиляються записи з некоректними/відсутніми обов'язковими полями або порушенням заданих доменних діапазонів. '?' у трьох nullable-полях не є причиною відхилення — він нормалізується до SQL NULL.


### 3.2. Перевірка потенційних дублікатів

У наборі немає ідентифікатора людини. Тому природного business key, який гарантує унікальність особи, немає. Перевіряємо повний збіг усіх 15 предметних полів (14 ознак + target), тобто точні дублікати рядків, а не доводимо, що це одна й та сама людина.

In [ ]:
duplicate_audit = query_df('''
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT ROW(
            age, workclass, fnlwgt, education, education_num,
            marital_status, occupation, relationship, race, sex,
            capital_gain, capital_loss, hours_per_week, native_country,
            income_over_50k
        )) AS distinct_full_records,
        COUNT(*) - COUNT(DISTINCT ROW(
            age, workclass, fnlwgt, education, education_num,
            marital_status, occupation, relationship, race, sex,
            capital_gain, capital_loss, hours_per_week, native_country,
            income_over_50k
        )) AS duplicate_rows
    FROM hw3_adult
''')
display(duplicate_audit)
dup_n = int(duplicate_audit.loc[0, 'duplicate_rows'])
print(
    f'Знайдено {dup_n:,} повторних входжень повністю однакових записів. '
    'Це сигнал для перевірки, але без person_id неможливо визначити, '
    'чи це дубль однієї особи, чи різні люди з однаковими атрибутами.'
)

,total_rows,distinct_full_records,duplicate_rows
0,32561,32537,24


Знайдено 24 повторних входжень повністю однакових записів. Це сигнал для перевірки, але без person_id неможливо визначити, чи це дубль однієї особи, чи різні люди з однаковими атрибутами.


### 3.3. Null-rate у clean і спеціальні маркери у staging

Перевіряємо щонайменше п'ять полів. У цьому наборі `?` є джерельним маркером відсутнього значення для `workclass`, `occupation` та `native_country`; у фінальній таблиці він стає SQL `NULL`. Інші категорії зі значенням `unknown` не вигадуємо й не перетворюємо без підстав.

In [ ]:
null_audit = query_df('''
    SELECT
        COUNT(*) AS total_rows,
        ROUND(100.0 * COUNT(*) FILTER (WHERE age IS NULL) / NULLIF(COUNT(*), 0), 2) AS age_null_pct,
        ROUND(100.0 * COUNT(*) FILTER (WHERE workclass IS NULL) / NULLIF(COUNT(*), 0), 2) AS workclass_null_pct,
        ROUND(100.0 * COUNT(*) FILTER (WHERE education IS NULL) / NULLIF(COUNT(*), 0), 2) AS education_null_pct,
        ROUND(100.0 * COUNT(*) FILTER (WHERE occupation IS NULL) / NULLIF(COUNT(*), 0), 2) AS occupation_null_pct,
        ROUND(100.0 * COUNT(*) FILTER (WHERE native_country IS NULL) / NULLIF(COUNT(*), 0), 2) AS native_country_null_pct,
        ROUND(100.0 * COUNT(*) FILTER (WHERE hours_per_week IS NULL) / NULLIF(COUNT(*), 0), 2) AS hours_null_pct
    FROM hw3_adult
''')
display(null_audit)

sentinel_audit = query_df('''
    SELECT
        COUNT(*) FILTER (WHERE BTRIM(workclass_raw) = '?') AS workclass_question_mark,
        COUNT(*) FILTER (WHERE BTRIM(occupation_raw) = '?') AS occupation_question_mark,
        COUNT(*) FILTER (WHERE BTRIM(native_country_raw) = '?') AS native_country_question_mark,
        COUNT(*) FILTER (WHERE NULLIF(BTRIM(workclass_raw), '') IS NULL) AS workclass_blank_or_null,
        COUNT(*) FILTER (WHERE NULLIF(BTRIM(occupation_raw), '') IS NULL) AS occupation_blank_or_null,
        COUNT(*) FILTER (WHERE NULLIF(BTRIM(native_country_raw), '') IS NULL) AS native_country_blank_or_null,
        COUNT(*) FILTER (WHERE LOWER(BTRIM(workclass_raw)) = 'unknown') AS workclass_unknown_literal
    FROM hw3_adult_staging
''')
display(sentinel_audit)
print(
    'Порівняйте null-rate у clean з кількістю маркерів у staging: '
    'для цих nullable-полів очікується, що основна частина пропусків '
    "походить саме з '?'. Нульовий null-rate в інших колонках означає лише "
    'відсутність SQL NULL після правил завантаження, а не безумовну '
    'коректність усіх значень.'
)

,total_rows,age_null_pct,workclass_null_pct,education_null_pct,occupation_null_pct,native_country_null_pct,hours_null_pct
0,32561,0.00,5.64,0.00,5.66,1.79,0.00


,workclass_question_mark,occupation_question_mark,native_country_question_mark,workclass_blank_or_null,occupation_blank_or_null,native_country_blank_or_null,workclass_unknown_literal
0,1836,1843,583,0,0,0,0


Порівняйте null-rate у clean з кількістю маркерів у staging: для цих nullable-полів очікується, що основна частина пропусків походить саме з '?'. Нульовий null-rate в інших колонках означає лише відсутність SQL NULL після правил завантаження, а не безумовну коректність усіх значень.


## 6. Завдання 4 — Розподіли та потенційні аномалії

### 4.1. Розподіл категоріальної колонки `workclass`

In [ ]:
workclass_dist = query_df('''
    SELECT
        COALESCE(workclass, '[NULL / missing]') AS workclass_category,
        COUNT(*) AS n_rows,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_rows
    FROM hw3_adult
    GROUP BY workclass
    ORDER BY n_rows DESC, workclass_category
    LIMIT 10
''')
display(workclass_dist)
if not workclass_dist.empty:
    top = workclass_dist.iloc[0]
    print(
        f"Найбільша група — {top['workclass_category']}: "
        f"{int(top['n_rows']):,} рядків ({top['pct_rows']}%). "
        'Сильна концентрація в одній категорії може впливати на поведінку '
        'моделі; рідкісні категорії та пропуски варто контролювати окремо.'
    )

,workclass_category,n_rows,pct_rows
0,Private,22696,69.70
1,Self-emp-not-inc,2541,7.80
2,Local-gov,2093,6.43
3,[NULL / missing],1836,5.64
4,State-gov,1298,3.99
5,Self-emp-inc,1116,3.43
6,Federal-gov,960,2.95
7,Without-pay,14,0.04
8,Never-worked,7,0.02


Найбільша група — Private: 22,696 рядків (69.70%). Сильна концентрація в одній категорії може впливати на поведінку моделі; рідкісні категорії та пропуски варто контролювати окремо.


### 4.2. Базова статистика числової колонки `hours_per_week`

In [ ]:
hours_stats = query_df('''
    SELECT
        COUNT(hours_per_week) AS n_non_null,
        MIN(hours_per_week) AS min_value,
        MAX(hours_per_week) AS max_value,
        ROUND(AVG(hours_per_week)::NUMERIC, 2) AS avg_value,
        ROUND(STDDEV_SAMP(hours_per_week)::NUMERIC, 2) AS sd_value
    FROM hw3_adult
''')
display(hours_stats)
s = hours_stats.iloc[0]
print(
    f"Для {int(s['n_non_null']):,} непорожніх значень кількість годин "
    f"лежить у діапазоні {s['min_value']}–{s['max_value']}; "
    f"середнє — {s['avg_value']}, вибіркове стандартне відхилення — {s['sd_value']}. "
    'Середнє не описує форму розподілу повністю, тому його слід читати разом '
    'із частотами годинних груп.'
)

,n_non_null,min_value,max_value,avg_value,sd_value
0,32561,1,99,40.44,12.35


Для 32,561 непорожніх значень кількість годин лежить у діапазоні 1–99; середнє — 40.44, вибіркове стандартне відхилення — 12.35. Середнє не описує форму розподілу повністю, тому його слід читати разом із частотами годинних груп.


### 4.3. Статистичний outlier-check за правилом 3σ

Правило 3σ застосовуємо до `hours_per_week` як просту первинну перевірку. Воно припускає приблизно симетричний розподіл, тоді як реальні робочі години можуть мати асиметрію та скупчення на типових значеннях. Тому знайдені рядки не видаляємо автоматично.

In [ ]:
outliers = query_df('''
    WITH stats AS (
        SELECT AVG(hours_per_week) AS mean_value,
               STDDEV_SAMP(hours_per_week) AS sd_value
        FROM hw3_adult
    )
    SELECT
        a.id, a.age, a.workclass, a.occupation,
        a.hours_per_week, a.income_over_50k,
        ROUND(((a.hours_per_week - s.mean_value) / NULLIF(s.sd_value, 0))::NUMERIC, 2) AS z_score
    FROM hw3_adult AS a
    CROSS JOIN stats AS s
    WHERE s.sd_value > 0
      AND ABS(a.hours_per_week - s.mean_value) > 3 * s.sd_value
    ORDER BY ABS(a.hours_per_week - s.mean_value) DESC
    LIMIT 20
''')
display(outliers)
print(
    f'Запит повернув {len(outliers)} рядків (обмеження LIMIT 20) із відхиленням понад 3σ. '
    'Великі значення годин можуть бути реальними. Рекомендована дія — прапорець '
    'для аналізу та перевірка джерела/розподілу; не видаляти їх лише через великий z-score.'
)

,id,age,workclass,occupation,hours_per_week,income_over_50k,z_score
0,10146,35,None,None,99,False,4.74
1,10269,56,Self-emp-inc,Transport-moving,99,False,4.74
2,9814,28,Private,Prof-specialty,99,False,4.74
3,9834,67,Private,Prof-specialty,99,False,4.74
4,8658,30,Private,Protective-serv,99,False,4.74
5,5436,44,Private,Prof-specialty,99,False,4.74
6,6902,53,Self-emp-not-inc,Transport-moving,99,False,4.74
7,8800,39,Private,Transport-moving,99,True,4.74
8,8397,50,Self-emp-inc,Exec-managerial,99,True,4.74
9,8076,44,Self-emp-not-inc,Other-service,99,False,4.74


Запит повернув 20 рядків (обмеження LIMIT 20) із відхиленням понад 3σ. Великі значення годин можуть бути реальними. Рекомендована дія — прапорець для аналізу та перевірка джерела/розподілу; не видаляти їх лише через великий z-score.


## 7. Завдання 5 — Десять DQL/EDA-запитів

Кожен блок містить бізнес-питання, SQL і інтерпретацію, що використовує повернуті PostgreSQL результати.

### Запит 1. Приклади записів із доходом понад $50K

**Business-question:** Які приклади людей у вибірці мають target доходу понад $50K?

In [ ]:
sql_q1 = '''
SELECT age, education, education_num, occupation,
       hours_per_week, native_country, income_over_50k
FROM hw3_adult
WHERE income_over_50k IS TRUE
ORDER BY age DESC, hours_per_week DESC
LIMIT 10
'''
q1 = query_df(sql_q1)
display(q1)
print(
    f'Показано {len(q1)} прикладів із target TRUE. Це лише верхні 10 рядків '
    'за сортуванням віку та годин, а не репрезентативна вибірка й не оцінка частки класу.'
)

,age,education,education_num,occupation,hours_per_week,native_country,income_over_50k
0,90,Prof-school,15,Prof-specialty,72,United-States,True
1,90,Masters,14,Exec-managerial,60,United-States,True
2,90,HS-grad,9,Craft-repair,50,Ecuador,True
3,90,Masters,14,Exec-managerial,50,United-States,True
4,90,HS-grad,9,None,40,United-States,True
5,90,Bachelors,13,Exec-managerial,40,England,True
6,90,Masters,14,Prof-specialty,40,United-States,True
7,90,Bachelors,13,Sales,15,United-States,True
8,84,Some-college,10,Sales,35,United-States,True
9,83,Bachelors,13,Exec-managerial,55,United-States,True


Показано 10 прикладів із target TRUE. Це лише верхні 10 рядків за сортуванням віку та годин, а не репрезентативна вибірка й не оцінка частки класу.


### Запит 2. Працюючі години або приріст капіталу у віці 25–60 років

**Business-question:** Скільки записів віком 25–60 років мають щонайменше 50 робочих годин або ненульовий приріст капіталу, якщо одночасно не відсутні і `workclass`, і `occupation`?

In [ ]:
sql_q2 = '''
SELECT
    COUNT(*) AS matched_rows,
    ROUND(100.0 * COUNT(*) / NULLIF((SELECT COUNT(*) FROM hw3_adult), 0), 2) AS pct_of_clean
FROM hw3_adult
WHERE age BETWEEN 25 AND 60
  AND (hours_per_week >= 50 OR capital_gain > 0)
  AND NOT (workclass IS NULL AND occupation IS NULL)
'''
q2 = query_df(sql_q2)
display(q2)
r = q2.iloc[0]
print(
    f"Умову виконують {int(r['matched_rows']):,} рядків, або {r['pct_of_clean']}% clean-таблиці. "
    'Дужки визначають, що достатньо однієї з двох умов щодо годин/приросту капіталу; '
    'NOT виключає записи, де одночасно бракує обох категорій.'
)

,matched_rows,pct_of_clean
0,7123,21.88


Умову виконують 7,123 рядків, або 21.88% clean-таблиці. Дужки визначають, що достатньо однієї з двох умов щодо годин/приросту капіталу; NOT виключає записи, де одночасно бракує обох категорій.


### Запит 3. Рівні освіти з найбільш релевантних категорій

**Business-question:** Як розподілені записи між вибраними категоріями освіти?

In [ ]:
sql_q3 = '''
SELECT education, COUNT(*) AS n_rows,
       ROUND(100.0 * COUNT(*) / NULLIF((SELECT COUNT(*) FROM hw3_adult), 0), 2) AS pct_of_all
FROM hw3_adult
WHERE education IN ('Bachelors', 'Masters', 'Doctorate', 'HS-grad', 'Some-college')
GROUP BY education
ORDER BY n_rows DESC, education
'''
q3 = query_df(sql_q3)
display(q3)
if not q3.empty:
    r = q3.iloc[0]
    print(
        f"Серед вибраних рівнів найчастіший — {r['education']} "
        f"({int(r['n_rows']):,} рядків; {r['pct_of_all']}% від усієї clean-таблиці). "
        'Ці частки рахуються від усієї таблиці, а не лише від п’яти вибраних категорій.'
    )

,education,n_rows,pct_of_all
0,HS-grad,10501,32.25
1,Some-college,7291,22.39
2,Bachelors,5355,16.45
3,Masters,1723,5.29
4,Doctorate,413,1.27


Серед вибраних рівнів найчастіший — HS-grad (10,501 рядків; 32.25% від усієї clean-таблиці). Ці частки рахуються від усієї таблиці, а не лише від п’яти вибраних категорій.


### Запит 4. Країни, назва яких містить `united`

**Business-question:** Яка кількість записів має країну народження, назва якої містить підрядок `united` без урахування регістру?

In [ ]:
sql_q4 = '''
SELECT native_country, COUNT(*) AS n_rows,
       ROUND(100.0 * COUNT(*) / NULLIF((SELECT COUNT(*) FROM hw3_adult), 0), 2) AS pct_of_all
FROM hw3_adult
WHERE native_country ILIKE '%united%'
GROUP BY native_country
ORDER BY n_rows DESC
'''
q4 = query_df(sql_q4)
display(q4)
matched = int(q4['n_rows'].sum()) if not q4.empty else 0
print(
    f"Шаблону ILIKE відповідає {matched:,} рядків. ILIKE шукає фрагмент назви без "
    'урахування регістру; це корисно для пошуку, але не замінює контрольований довідник кодів країн.'
)

,native_country,n_rows,pct_of_all
0,United-States,29170,89.59


Шаблону ILIKE відповідає 29,170 рядків. ILIKE шукає фрагмент назви без урахування регістру; це корисно для пошуку, але не замінює контрольований довідник кодів країн.


### Запит 5. Пропуски у важливих категоріях

**Business-question:** Скільки записів мають заповнену та незаповнену професію, тип зайнятості й країну народження?

In [ ]:
sql_q5 = '''
SELECT 'occupation' AS field,
       COUNT(*) FILTER (WHERE occupation IS NULL) AS missing_rows,
       COUNT(*) FILTER (WHERE occupation IS NOT NULL) AS present_rows
FROM hw3_adult
UNION ALL
SELECT 'workclass',
       COUNT(*) FILTER (WHERE workclass IS NULL),
       COUNT(*) FILTER (WHERE workclass IS NOT NULL)
FROM hw3_adult
UNION ALL
SELECT 'native_country',
       COUNT(*) FILTER (WHERE native_country IS NULL),
       COUNT(*) FILTER (WHERE native_country IS NOT NULL)
FROM hw3_adult
ORDER BY field
'''
q5 = query_df(sql_q5)
display(q5)
print(
    'Наявність пропусків у категоріальних полях може бути інформативною сама по собі, '
    'але не слід без перевірки трактувати її як окрему категорію фактичного стану людини. '
    'Для ML можна додати missing-indicator або застосувати імпутацію, порівнявши якість обох підходів.'
)

,field,missing_rows,present_rows
0,native_country,583,31978
1,occupation,1843,30718
2,workclass,1836,30725


Наявність пропусків у категоріальних полях може бути інформативною сама по собі, але не слід без перевірки трактувати її як окрему категорію фактичного стану людини. Для ML можна додати missing-indicator або застосувати імпутацію, порівнявши якість обох підходів.


### Запит 6. Розподіл зайнятості з явною категорією пропуску

**Business-question:** Які категорії `workclass` найпоширеніші, якщо порожнє текстове значення додатково нормалізувати через `NULLIF`, а `NULL` показати окремо через `COALESCE`?

In [ ]:
sql_q6 = '''
SELECT COALESCE(NULLIF(BTRIM(workclass), ''), '[missing]') AS workclass_group,
       COUNT(*) AS n_rows
FROM hw3_adult
GROUP BY COALESCE(NULLIF(BTRIM(workclass), ''), '[missing]')
ORDER BY n_rows DESC, workclass_group
LIMIT 10
'''
q6 = query_df(sql_q6)
display(q6)
if not q6.empty:
    r = q6.iloc[0]
    print(
        f"Найбільша група — {r['workclass_group']} ({int(r['n_rows']):,} рядків). "
        'Мітка [missing] створена лише для звіту; вона не записується назад у таблицю '
        'і не змішується з реальними категоріями workclass.'
    )

,workclass_group,n_rows
0,Private,22696
1,Self-emp-not-inc,2541
2,Local-gov,2093
3,[missing],1836
4,State-gov,1298
5,Self-emp-inc,1116
6,Federal-gov,960
7,Without-pay,14
8,Never-worked,7


Найбільша група — Private (22,696 рядків). Мітка [missing] створена лише для звіту; вона не записується назад у таблицю і не змішується з реальними категоріями workclass.


### Запит 7. Усі записи, які не є `Private`, включно з NULL

**Business-question:** Скільки записів мають тип зайнятості, відмінний від `Private`, або не мають його взагалі?

In [ ]:
sql_q7 = '''
SELECT
    COUNT(*) FILTER (WHERE workclass IS DISTINCT FROM 'Private') AS not_private_or_missing,
    COUNT(*) FILTER (WHERE workclass IS NOT DISTINCT FROM 'Private') AS private_rows,
    COUNT(*) AS total_rows
FROM hw3_adult
'''
q7 = query_df(sql_q7)
display(q7)
r = q7.iloc[0]
print(
    f"{int(r['not_private_or_missing']):,} рядків ({100 * r['not_private_or_missing'] / r['total_rows']:.2f}%) "
    'не дорівнюють Private або мають NULL. IS DISTINCT FROM відрізняється від звичайного <> тим, '
    'що порівнює NULL як окремий стан і повертає однозначний TRUE/FALSE.'
)

,not_private_or_missing,private_rows,total_rows
0,9865,22696,32561


9,865 рядків (30.30%) не дорівнюють Private або мають NULL. IS DISTINCT FROM відрізняється від звичайного <> тим, що порівнює NULL як окремий стан і повертає однозначний TRUE/FALSE.


### Запит 8. Унікальні значення країни народження

**Business-question:** Які різні значення країни народження зустрічаються у заповнених записах?

In [ ]:
sql_q8 = '''
SELECT DISTINCT native_country
FROM hw3_adult
WHERE native_country IS NOT NULL
ORDER BY native_country
'''
q8 = query_df(sql_q8)
display(q8)
print(
    f"У clean-таблиці знайдено {len(q8)} унікальних непорожніх значень native_country. "
    'DISTINCT показує перелік категорій, а не частоту кожної; для частот потрібен GROUP BY.'
)

,native_country
0,Cambodia
1,Canada
2,China
3,Columbia
4,Cuba
5,Dominican-Republic
6,Ecuador
7,El-Salvador
8,England
9,France


У clean-таблиці знайдено 41 унікальних непорожніх значень native_country. DISTINCT показує перелік категорій, а не частоту кожної; для частот потрібен GROUP BY.


### Запит 9. Найбільші значення приросту капіталу

**Business-question:** Які десять записів мають найбільші значення `capital_gain`?

In [ ]:
sql_q9 = '''
SELECT id, age, education, occupation, capital_gain, capital_loss,
       hours_per_week, income_over_50k
FROM hw3_adult
WHERE capital_gain > 0
ORDER BY capital_gain DESC, id
LIMIT 10
'''
q9 = query_df(sql_q9)
display(q9)
if not q9.empty:
    print(
        f"У топ-10 максимальне значення capital_gain дорівнює {int(q9.iloc[0]['capital_gain']):,}. "
        'Це крайні спостереження, але саме по собі велике значення не є помилкою: '
        'їх потрібно перевірити за документацією та контекстом, а не видаляти автоматично.'
    )
else:
    print('У таблиці немає додатних значень capital_gain.')

,id,age,education,occupation,capital_gain,capital_loss,hours_per_week,income_over_50k
0,1247,54,Prof-school,Prof-specialty,99999,0,60,True
1,1370,52,HS-grad,Exec-managerial,99999,0,40,True
2,1485,53,HS-grad,Sales,99999,0,40,True
3,1531,52,Bachelors,Exec-managerial,99999,0,50,True
4,1619,46,Prof-school,Prof-specialty,99999,0,60,True
5,1685,43,Bachelors,Exec-managerial,99999,0,60,True
6,1768,49,Bachelors,Prof-specialty,99999,0,20,True
7,1774,50,Some-college,Exec-managerial,99999,0,50,True
8,1829,22,HS-grad,Prof-specialty,99999,0,40,True
9,2106,43,Some-college,Sales,99999,0,55,True


У топ-10 максимальне значення capital_gain дорівнює 99,999. Це крайні спостереження, але саме по собі велике значення не є помилкою: їх потрібно перевірити за документацією та контекстом, а не видаляти автоматично.


### Запит 10. Частка класу `>50K` за діапазонами робочих годин

**Business-question:** Як відрізняється частка записів із доходом понад $50K між діапазонами годин роботи на тиждень?

In [ ]:
sql_q10 = '''
SELECT
    CASE
        WHEN hours_per_week BETWEEN 1 AND 20 THEN '01-20'
        WHEN hours_per_week BETWEEN 21 AND 39 THEN '21-39'
        WHEN hours_per_week = 40 THEN '40'
        WHEN hours_per_week BETWEEN 41 AND 60 THEN '41-60'
        ELSE '61+'
    END AS hours_band,
    COUNT(*) AS n_rows,
    ROUND(AVG(hours_per_week)::NUMERIC, 2) AS avg_hours,
    ROUND(100.0 * AVG(income_over_50k::INTEGER), 2) AS pct_income_over_50k
FROM hw3_adult
GROUP BY 1
ORDER BY MIN(hours_per_week)
'''
q10 = query_df(sql_q10)
display(q10)
if not q10.empty:
    top_rate = q10.sort_values('pct_income_over_50k', ascending=False).iloc[0]
    print(
        f"Найвища частка target TRUE спостерігається у групі {top_rate['hours_band']} "
        f"({top_rate['pct_income_over_50k']}%; {int(top_rate['n_rows']):,} рядків). "
        'Це описова асоціація, а не доказ того, що збільшення робочих годин спричиняє вищий дохід.'
    )

,hours_band,n_rows,avg_hours,pct_income_over_50k
0,01-20,2928,15.13,6.66
1,21-39,4835,31.61,11.23
2,40,15217,40.00,21.34
3,41-60,8471,50.55,40.75
4,61+,1110,74.49,36.40


Найвища частка target TRUE спостерігається у групі 41-60 (40.75%; 8,471 рядків). Це описова асоціація, а не доказ того, що збільшення робочих годин спричиняє вищий дохід.


## 8. Завдання 6 — Reflection

У наборі Adult основна проблема якості — спеціальний маркер `?` у полях `workclass`, `occupation` і `native_country`. Я нормалізував його до SQL `NULL`, оскільки він позначає відсутнє значення, а не окрему професію чи країну. Решту категорій залишив без довільного перекодування. Перевірка точних дублікатів використовує всі предметні поля, але не дозволяє ідентифікувати повтори конкретної людини: у наборі немає персонального ключа. Додатково важливо перевіряти діапазони числових полів і не вважати кожне екстремальне значення помилкою.

Для базової моделі можна розглядати `age`, `education_num`, `workclass`, `occupation`, `marital_status`, `relationship`, `capital_gain`, `capital_loss` і `hours_per_week`. `education` частково дублює `education_num`, тому варто порівняти варіанти й уникати зайвої надлишковості. `fnlwgt` насамперед є вагою спостереження для аналізу репрезентативності, а не звичайною персональною ознакою. `race` і `sex` потребують особливої обережності: їх доцільно використовувати для окремого fairness-аудиту, а не автоматично включати в production-модель.

`income_over_50k` є target і не може бути feature. Явного post-event поля на кшталт тривалості дзвінка тут немає, однак `capital_gain` і `capital_loss` тісно пов'язані з формуванням доходу, тому потрібно чітко визначити момент прогнозу та перевірити ризик концептуального leakage. Pipeline варто доповнити схемою версій, журналом відхилених рядків, метриками пропусків і перевірками допустимих категорій. Природна ML-задача — бінарна класифікація. Водночас дані походять із перепису 1994 року, тож їх не можна без перевірки переносити на сучасне населення.

**Підсумок:** цільова задача — binary classification; результати аудиту та запитів вище дають базову оцінку придатності даних, але не замінюють перевірку репрезентативності й справедливості моделі.

## 9. Підсумкова перевірка виконання

Клітинка нижче перевіряє, що staging і clean існують, clean має щонайменше 30 000 рядків, а всі 15 колонок staging завантажено.

In [ ]:
final_check = query_df('''
    SELECT
        (SELECT COUNT(*) FROM hw3_adult_staging) AS staging_rows,
        (SELECT COUNT(*) FROM hw3_adult) AS clean_rows,
        (SELECT COUNT(*) FROM information_schema.columns
         WHERE table_schema = 'public' AND table_name = 'hw3_adult_staging') AS staging_columns,
        (SELECT COUNT(*) FROM information_schema.columns
         WHERE table_schema = 'public' AND table_name = 'hw3_adult') AS clean_columns
''')
display(final_check)
assert int(final_check.loc[0, 'staging_rows']) >= 30_000
assert int(final_check.loc[0, 'clean_rows']) >= 30_000
assert int(final_check.loc[0, 'staging_columns']) == 15
assert int(final_check.loc[0, 'clean_columns']) == 16  # id + 15 предметних полів
print('Перевірки пройдено. PostgreSQL 16, Psycopg 3 і всі SQL-клітинки виконані успішно.')
conn.close()
print("З’єднання з PostgreSQL коректно закрито.")


,staging_rows,clean_rows,staging_columns,clean_columns
0,32561,32561,15,16


Перевірки пройдено. PostgreSQL 16, Psycopg 3 і всі SQL-клітинки виконані успішно.
З’єднання з PostgreSQL коректно закрито.
